# Baseline Modeling & Pipeline Construction
**Objective:** Establish a reproducible evaluation environment, construct a robust preprocessing pipeline for mixed data types, and evaluate classical baseline models.

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## 1. Data Ingestion & Train/Test Split
Isolating a 20% holdout set to prevent data leakage during stateful preprocessing.

In [6]:
# Load engineered data
df = pd.read_csv('../data/interim/used_cars_engineered.csv')

# Ensure target leakage is prevented
columns_to_drop = ['AskPrice_Log']
if 'AskPrice' in df.columns:
    columns_to_drop.append('AskPrice')

X = df.drop(columns=columns_to_drop)
y = df['AskPrice_Log']

# Reproducible split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (10921, 6)
Testing features shape: (2731, 6)


## 2. Preprocessing Engine
Constructing a `ColumnTransformer` to handle nominal categoricals, ordinal categoricals, and numerical scaling independently.

In [7]:
# Feature categorization
nominal_cols = ['Brand', 'FuelType', 'Transmission']
ordinal_cols = ['Owner']
num_cols = ['Age', 'kmDriven']

# Define explicit ordinal mapping to preserve monotonic depreciation logic
owner_categories = [['first', 'second']]

# Construct ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols),
        ('ord', OrdinalEncoder(categories=owner_categories, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('num', StandardScaler(), num_cols)
    ],
    remainder='drop'
)

## 3. Evaluation Utility
Helper function to calculate actual currency errors from log-transformed predictions.

In [8]:
def evaluate_model(pipeline, X_test, y_test, model_name="Model"):
    """Predicts, applies inverse log transformation, and prints evaluation metrics."""
    # Predict in log space
    y_pred_log = pipeline.predict(X_test)
    
    # Inverse transform to actual currency (INR)
    y_pred_actual = np.expm1(y_pred_log)
    y_test_actual = np.expm1(y_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test_actual, y_pred_actual)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
    
    print(f"--- {model_name} ---")
    print(f"MAE:  ₹{mae:,.2f}")
    print(f"RMSE: ₹{rmse:,.2f}\n")

## 4. Baseline Establishment
Evaluating a `DummyRegressor` to establish the absolute error floor, followed by a `LinearRegression` model to validate feature signal.

In [9]:
# 1. Dummy Baseline (Mean strategy)
dummy_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))
])
dummy_pipeline.fit(X_train, y_train)
evaluate_model(dummy_pipeline, X_test, y_test, "Dummy Regressor (Mean)")

# 2. Linear Regression Baseline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
lr_pipeline.fit(X_train, y_train)
evaluate_model(lr_pipeline, X_test, y_test, "Linear Regression")

--- Dummy Regressor (Mean) ---
MAE:  ₹679,072.11
RMSE: ₹1,777,147.53

--- Linear Regression ---
MAE:  ₹349,276.59
RMSE: ₹1,222,613.13

